# `ChatGeneration: Generation`

Represents a single structured output from a chat model.

The message is usually an `AIMessage`. Users generally access chat-model results through an `AIMessage` returned by runnable interfaces or through an `LLMResult` available in callbacks.

## Fields

```python
text: str = "" # Text extracted from the output message; should not be set directly
message: BaseMessage # Structured message produced by the chat model
type: Literal["ChatGeneration"] = "ChatGeneration" # Discriminator used exclusively for serialization
```

The class also inherits generation metadata from `Generation`.

## Methods

### `set_text`

Sets `text` from the contents of `message`.

```python
@model_validator(mode="after")
set_text(
    self,
) -> Self # Validated generation with text populated from the message
```

For list content containing legacy dictionaries with a `"text"` key and no `"type"` field, it concatenates strings and text blocks. Otherwise, it delegates text extraction to `message.text`.

Raises `ValueError` if the message content is not a supported string or list representation.

In [1]:
from langchain_core.messages import AIMessage # Import the chat message class
from langchain_core.outputs import ChatGeneration # Import the chat generation class

message1 = AIMessage(content="Hello, Saad!") # Create a message containing normal text
generation1 = ChatGeneration(message=message1) # Create a chat generation from the message

print("Message content:", generation1.message.content) # Display the original message content
print("Generated text:", generation1.text) # Display the automatically extracted text

message2 = AIMessage( # Create a message containing legacy text blocks
    content=[ # Start the list of content blocks
        "Hello, ", # Add a normal string block
        {"text": "how are you?"}, # Add a legacy dictionary text block
    ] # End the list of content blocks
) # Finish creating the second message

generation2 = ChatGeneration(message=message2) # Create another chat generation

print("Legacy content:", generation2.message.content) # Display the original list content
print("Combined text:", generation2.text) # Display the automatically joined text

Message content: Hello, Saad!
Generated text: Hello, Saad!
Legacy content: ['Hello, ', {'text': 'how are you?'}]
Combined text: Hello, how are you?


# `ChatGenerationChunk: ChatGeneration`

Represents a streamable chat-generation chunk that can be concatenated with other chunks.

## Fields

```python
message: BaseMessageChunk # Message chunk produced by the chat model
type: Literal["ChatGenerationChunk"] = "ChatGenerationChunk" # Discriminator used exclusively for serialization
```

It inherits `text` and generation metadata from `ChatGeneration`.

## Methods

### `__add__`

Concatenates this chunk with another chunk or a list of chunks.

```python
__add__(
    self,
    other: ChatGenerationChunk | list[ChatGenerationChunk], # Chunk or chunks to concatenate
) -> ChatGenerationChunk # Newly concatenated chunk
```

Message chunks are concatenated in order. Non-empty `generation_info` dictionaries are merged, and the result uses `None` when the merged metadata is empty.

Raises `TypeError` when `other` is neither a `ChatGenerationChunk` nor a list containing only `ChatGenerationChunk` objects.

---

# `merge_chat_generation_chunks`

Merges a list of chat-generation chunks into one chunk.

```python
merge_chat_generation_chunks(
    chunks: list[ChatGenerationChunk], # Chunks to merge in order
) -> ChatGenerationChunk | None # Merged chunk, or None when the list is empty
```

Returns the only chunk unchanged when the list contains one element. For multiple elements, it concatenates the first chunk with the remaining chunks.

In [ ]:
from langchain_core.messages import AIMessageChunk # Import the streaming message chunk class
from langchain_core.outputs import ChatGenerationChunk # Import the generation chunk class
from langchain_core.outputs.chat_generation import merge_chat_generation_chunks # Import the merge function

chunk1 = ChatGenerationChunk( # Create the first streamed chunk
    message=AIMessageChunk(content="Hello "), # Store the first piece of text
    generation_info={"part": 1}, # Store metadata for the first chunk
) # Finish creating the first chunk

chunk2 = ChatGenerationChunk( # Create the second streamed chunk
    message=AIMessageChunk(content="Saad!"), # Store the second piece of text
    generation_info={"status": "finished"}, # Store metadata for the second chunk
) # Finish creating the second chunk

combined_with_add = chunk1 + chunk2 # Combine chunks using the + operator

print("Using + operator:", combined_with_add.text) # Display the combined text
print("Merged metadata:", combined_with_add.generation_info) # Display merged metadata

chunks = [chunk1, chunk2] # Store the chunks in order
combined_with_function = merge_chat_generation_chunks(chunks) # Merge the complete list

if combined_with_function is not None: # Ensure the result exists
    print("Using merge function:", combined_with_function.text) # Display merged text
    print("Final message:", combined_with_function.message.content) # Display merged content

empty_result = merge_chat_generation_chunks([]) # Merge an empty list
print("Empty list result:", empty_result) # Display None